# Spatial & Persistent Structures

Two families an array or a plain BST can't serve well. **Spatial** structures answer *geometric* questions — "which intervals cover this point?", "what's the nearest point to here?", "what's inside this box?" — where a 1-D sorted order (the **trees** notebook's BST) simply has no axis to sort on. **Persistent** structures answer the *time* question — keep **every past version** queryable after each update, instead of overwriting state in place. Both fall out of the same move: **augment a tree** and, for persistence, **copy only the path you touch** so old versions keep sharing the rest.

**Contents**

1. **The problem** — why arrays and BSTs run out
2. **Interval tree** — intervals overlapping a point or a range
3. **k-d tree** — 2-D nearest-neighbour & orthogonal range search
4. **Persistent segment tree** — versioned range-sum via path copying
5. **When to use what**
6. **Complexity cheat-sheet**


## 1. The problem — why arrays and BSTs run out

A sorted array plus `bisect` (the **trees** notebook) is the right tool when your data lives on **one axis** and you overwrite freely. Three kinds of question break that model:

- **Overlap, not equality.** Given a set of intervals `[lo, hi]`, which ones *cover* a point `p`? There's no single key to sort by — an interval matches on a relationship between *two* coordinates, so a BST keyed on `lo` (or `hi`) can't prune.
- **More than one dimension.** "Nearest point to `(x, y)`" or "all points in this rectangle" needs the structure to branch on **x sometimes and y other times**. A 1-D order throws away one coordinate.
- **History.** A normal update **destroys** the old value. If you need to query the array *as it was* at version 3 while version 7 exists, in-place mutation can't help.

The fixes share a spine: take a balanced binary tree and **augment** it (store extra summaries per node so you can prune), and — for history — **path-copy** on update so each version is a new root over mostly-shared nodes. Sums and ranges here build directly on the **fenwick & segment trees** notebook.

## 2. Interval tree — overlap queries

Store a set of closed intervals `[lo, hi]`; answer **"which intervals overlap a query point `p`"** and **"which overlap a query interval `[qlo, qhi]`"**. The trick is a **centered** interval tree: pick a `center` coordinate, and split the intervals into three groups — those entirely **left** of center, those entirely **right**, and those that **cross** it. Recurse on left/right; the crossing set lives at this node, stored **twice** — once sorted by `lo` ascending, once by `hi` descending.

That double-sort is what makes a point query O(log n + k):

- if `p < center`, only crossing intervals with `lo ≤ p` can match — scan `by_lo` and **stop at the first miss**, then recurse left only.
- if `p > center`, only those with `hi ≥ p` can match — scan `by_hi` and stop early, recurse right only.
- if `p == center`, every crossing interval matches.

In [1]:
class IntervalTree:
    """Centered interval tree over closed intervals [lo, hi]."""
    class _Node:
        __slots__ = ("center", "left", "right", "by_lo", "by_hi")
        def __init__(self, center):
            self.center = center
            self.left = self.right = None
            self.by_lo = []          # crossing intervals, sorted by lo ascending
            self.by_hi = []          # same set, sorted by hi descending

    def __init__(self, intervals):
        self.root = self._build(list(intervals))

    def _build(self, ivs):
        if not ivs:
            return None
        mids = sorted((lo + hi) / 2 for lo, hi in ivs)
        node = self._Node(mids[len(mids) // 2])      # median midpoint = split point
        left, right, cross = [], [], []
        for lo, hi in ivs:
            if hi < node.center:   left.append((lo, hi))
            elif lo > node.center: right.append((lo, hi))
            else:                  cross.append((lo, hi))
        node.by_lo = sorted(cross, key=lambda iv: iv[0])
        node.by_hi = sorted(cross, key=lambda iv: iv[1], reverse=True)
        node.left, node.right = self._build(left), self._build(right)
        return node

    def query_point(self, p):                        # intervals covering point p
        out = []; self._qp(self.root, p, out); return out

    def _qp(self, node, p, out):
        if node is None:
            return
        if p < node.center:
            for lo, hi in node.by_lo:                # sorted by lo -> stop early
                if lo <= p: out.append((lo, hi))
                else:       break
            self._qp(node.left, p, out)
        elif p > node.center:
            for lo, hi in node.by_hi:                # sorted by hi desc -> stop early
                if hi >= p: out.append((lo, hi))
                else:       break
            self._qp(node.right, p, out)
        else:                                        # p == center: all cross
            out.extend(node.by_lo)
            self._qp(node.left, p, out); self._qp(node.right, p, out)

    def query_interval(self, qlo, qhi):              # intervals overlapping [qlo, qhi]
        out = []; self._qi(self.root, qlo, qhi, out); return out

    def _qi(self, node, qlo, qhi, out):
        if node is None:
            return
        for lo, hi in node.by_lo:                    # overlap iff lo<=qhi and hi>=qlo
            if lo <= qhi and hi >= qlo: out.append((lo, hi))
        if qlo < node.center: self._qi(node.left, qlo, qhi, out)
        if qhi > node.center: self._qi(node.right, qlo, qhi, out)

it = IntervalTree([(1, 5), (3, 8), (6, 10), (4, 4), (9, 12)])
print("cover point 4   :", sorted(it.query_point(4)))     # [1,5],[3,8],[4,4]
print("cover point 7   :", sorted(it.query_point(7)))     # [3,8],[6,10]
print("overlap [9, 11] :", sorted(it.query_interval(9, 11)))   # [6,10],[9,12]


cover point 4   : [(1, 5), (3, 8), (4, 4)]
cover point 7   : [(3, 8), (6, 10)]
overlap [9, 11] : [(6, 10), (9, 12)]


**Proof vs brute force.** A centered tree is easy to get subtly wrong (early-stop conditions, the `p == center` case), so we hammer it: hundreds of random interval sets, each checked against the O(n) filter `lo ≤ p ≤ hi` for points and `lo ≤ qhi and hi ≥ qlo` for ranges.

In [2]:
import random
random.seed(7)

def brute_point(ivs, p):       return sorted(iv for iv in ivs if iv[0] <= p <= iv[1])
def brute_interval(ivs, a, b): return sorted(iv for iv in ivs if iv[0] <= b and iv[1] >= a)

for _ in range(400):
    ivs = [(a, a + random.randint(0, 20))
           for a in (random.randint(0, 40) for _ in range(random.randint(1, 30)))]
    tree = IntervalTree(ivs)
    p = random.randint(-5, 50)
    assert sorted(tree.query_point(p)) == brute_point(ivs, p)
    a = random.randint(-5, 45); b = a + random.randint(0, 15)
    assert sorted(tree.query_interval(a, b)) == brute_interval(ivs, a, b)
print("400 random interval-trees: point & range queries match brute force")


400 random interval-trees: point & range queries match brute force


## 3. k-d tree — points in *k* dimensions

A **k-d tree** is a BST for points in *k* dimensions (here `k = 2`). The one idea: **cycle the splitting dimension by depth** — split on *x* at the root, *y* one level down, *x* again, and so on. Each node's plane carves space in two, so a subtree owns a rectangular region. Build it like quickselect's cousin: at each level sort by the current axis and take the **median** as the node, giving a balanced tree.

That partitioning powers two queries arrays can't:

- **Nearest neighbour** — descend to the leaf region containing the target, then unwind, only crossing to the *far* side of a split when the splitting plane is closer than the best distance found so far (`diff² < best_dist²`). That pruning is what beats the O(n) scan.
- **Orthogonal range search** — collect every point inside an axis-aligned box, skipping whole subtrees that fall outside it.

In [3]:
class KDTree:
    """2-D k-d tree: nearest-neighbour and orthogonal (box) range search."""
    class _Node:
        __slots__ = ("point", "left", "right", "axis")
        def __init__(self, point, axis):
            self.point = point; self.axis = axis
            self.left = self.right = None

    def __init__(self, points):
        self.root = self._build(list(points), 0)

    def _build(self, pts, depth):
        if not pts:
            return None
        axis = depth % 2                      # alternate x, y, x, y, ...
        pts.sort(key=lambda p: p[axis])
        mid = len(pts) // 2                   # median -> balanced
        node = self._Node(pts[mid], axis)
        node.left  = self._build(pts[:mid], depth + 1)
        node.right = self._build(pts[mid + 1:], depth + 1)
        return node

    def nearest(self, target):                # closest stored point to target
        best = [None, float("inf")]           # [point, squared distance]
        self._nn(self.root, target, best)
        return best[0]

    def _nn(self, node, target, best):
        if node is None:
            return
        dx = node.point[0] - target[0]; dy = node.point[1] - target[1]
        d2 = dx * dx + dy * dy
        if d2 < best[1]:
            best[0], best[1] = node.point, d2
        diff = target[node.axis] - node.point[node.axis]
        near, far = (node.left, node.right) if diff < 0 else (node.right, node.left)
        self._nn(near, target, best)
        if diff * diff < best[1]:             # far side only if plane is close enough
            self._nn(far, target, best)

    def range_search(self, lo, hi):           # points inside box [lo, hi] (inclusive)
        out = []; self._range(self.root, lo, hi, out); return out

    def _range(self, node, lo, hi, out):
        if node is None:
            return
        x, y = node.point
        if lo[0] <= x <= hi[0] and lo[1] <= y <= hi[1]:
            out.append(node.point)
        ax = node.axis
        if lo[ax] <= node.point[ax]: self._range(node.left, lo, hi, out)
        if node.point[ax] <= hi[ax]: self._range(node.right, lo, hi, out)

kd = KDTree([(2, 3), (5, 4), (9, 6), (4, 7), (8, 1), (7, 2)])
print("nearest to (6, 3)   :", kd.nearest((6, 3)))           # (5, 4) or (7, 2)
print("points in [(4,1),(8,5)] :", sorted(kd.range_search((4, 1), (8, 5))))


nearest to (6, 3)   : (7, 2)
points in [(4,1),(8,5)] : [(5, 4), (7, 2), (8, 1)]


**Proof vs brute force.** Nearest-neighbour pruning is the part that's easy to get wrong — prune too aggressively and you miss the true nearest. We compare against the brute-force `min` over **squared** distance (ties are fine: we assert the *distance* matches, not point identity), and range search against the O(n) box filter, over hundreds of random point clouds.

In [4]:
import random
random.seed(11)

def d2(a, b): return (a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2
def brute_near(pts, t): return min(pts, key=lambda p: d2(p, t))
def brute_box(pts, lo, hi):
    return sorted(p for p in pts if lo[0] <= p[0] <= hi[0] and lo[1] <= p[1] <= hi[1])

for _ in range(400):
    pts = [(random.randint(0, 50), random.randint(0, 50))
           for _ in range(random.randint(1, 40))]
    kd = KDTree(pts)
    t = (random.randint(-5, 55), random.randint(-5, 55))
    assert d2(kd.nearest(t), t) == d2(brute_near(pts, t), t)   # same distance
    lo = (random.randint(0, 40), random.randint(0, 40))
    hi = (lo[0] + random.randint(0, 15), lo[1] + random.randint(0, 15))
    assert sorted(kd.range_search(lo, hi)) == brute_box(pts, lo, hi)
print("400 random k-d trees: nearest-neighbour & box queries match brute force")


400 random k-d trees: nearest-neighbour & box queries match brute force


## 4. Persistent segment tree — keep every version

A normal **segment tree** (the **fenwick & segment trees** notebook) updates a leaf and its `log n` ancestors **in place**, destroying the old tree. A **persistent** one keeps every version queryable. The mechanism is **path copying**: an update touches exactly the root-to-leaf path — `O(log n)` nodes — so *copy just those nodes* and let the copies point at the **untouched subtrees of the previous version**. The result is a brand-new root for the new version; the old root still describes the old array perfectly.

So each version costs only `O(log n)` *new* nodes, not a full `O(n)` copy — the rest is structurally shared. And because an update takes a base version as input, you can **branch** off any past version, not just the latest.

In [5]:
class PersistentSegTree:
    """Range-sum segment tree where each update yields a new, independent version."""
    class _Node:
        __slots__ = ("sum", "left", "right")
        def __init__(self, s=0, left=None, right=None):
            self.sum = s; self.left = left; self.right = right

    def __init__(self, n):
        self.n = n
        self.versions = [self._build(0, n - 1)]      # version 0: all zeros

    def _build(self, lo, hi):
        if lo == hi:
            return self._Node(0)
        mid = (lo + hi) // 2
        return self._Node(0, self._build(lo, mid), self._build(mid + 1, hi))

    def update(self, version, i, value):             # set a[i]=value atop `version`
        root = self._update(self.versions[version], 0, self.n - 1, i, value)
        self.versions.append(root)                   # -> a new version id
        return len(self.versions) - 1

    def _update(self, node, lo, hi, i, value):
        if lo == hi:
            return self._Node(value)                 # new leaf
        mid = (lo + hi) // 2
        if i <= mid:                                 # recurse left, SHARE right subtree
            left  = self._update(node.left, lo, mid, i, value)
            right = node.right
        else:                                        # recurse right, SHARE left subtree
            left  = node.left
            right = self._update(node.right, mid + 1, hi, i, value)
        return self._Node(left.sum + right.sum, left, right)   # copy only this node

    def range_sum(self, version, l, r):              # sum of a[l..r] in a past version
        return self._query(self.versions[version], 0, self.n - 1, l, r)

    def _query(self, node, lo, hi, l, r):
        if r < lo or hi < l:
            return 0
        if l <= lo and hi <= r:
            return node.sum
        mid = (lo + hi) // 2
        return (self._query(node.left,  lo, mid, l, r)
                + self._query(node.right, mid + 1, hi, l, r))

pst = PersistentSegTree(8)
v1 = pst.update(0, 2, 5)            # a[2]=5  atop v0
v2 = pst.update(v1, 5, 9)          # a[5]=9  atop v1
v3 = pst.update(v1, 2, 1)          # a[2]=1  atop v1 (branch off v1, not v2!)
print("v0 sum[0,7] :", pst.range_sum(0, 0, 7))   # 0   -- original still intact
print("v1 sum[0,7] :", pst.range_sum(v1, 0, 7))  # 5
print("v2 sum[0,7] :", pst.range_sum(v2, 0, 7))  # 14
print("v3 sum[0,7] :", pst.range_sum(v3, 0, 7))  # 1   -- independent branch off v1


v0 sum[0,7] : 0
v1 sum[0,7] : 5
v2 sum[0,7] : 14
v3 sum[0,7] : 1


**The path-copying claim, made concrete.** An update should allocate *only* the nodes on one root-to-leaf path and reuse everything else from the base version. We update the leftmost leaf and check that the new root's entire **right** subtree is the *same object* (`is`) as the old one — copied path, shared remainder.

In [6]:
pst = PersistentSegTree(8)
old = pst.versions[0]
v1  = pst.update(0, 0, 5)           # touch leftmost leaf -> path goes all-left
new = pst.versions[v1]

print("new root is a copy      :", new is not old)
print("right subtree is SHARED :", new.right is old.right)   # untouched -> reused
print("left subtree is copied  :", new.left is not old.left) # on the update path

# count distinct nodes reachable only via the new path vs total
def all_nodes(node, seen):
    if node is None or id(node) in seen: return
    seen.add(id(node)); all_nodes(node.left, seen); all_nodes(node.right, seen)

base = set(); all_nodes(old, base)
both = set(); all_nodes(old, both); all_nodes(new, both)
import math
print(f"nodes after 1 update: {len(both)} total, {len(both) - len(base)} new "
      f"(~path length = {int(math.log2(8)) + 1})")


new root is a copy      : True
right subtree is SHARED : True
left subtree is copied  : True
nodes after 1 update: 19 total, 4 new (~path length = 4)


**Proof vs brute force.** The real test: drive a long sequence of updates — each **branching off a randomly chosen earlier version** — while keeping a plain Python-list *snapshot* of every version. Then every version's range-sum must match its snapshot. This proves old versions stay correct and independent after later updates.

In [7]:
import random
random.seed(13)

n = 16
pst = PersistentSegTree(n)
snapshots = [[0] * n]                       # snapshots[v] = the array at version v
for _ in range(150):
    base = random.randint(0, len(snapshots) - 1)   # branch off ANY existing version
    i = random.randint(0, n - 1); val = random.randint(0, 99)
    snap = snapshots[base][:]; snap[i] = val
    v = pst.update(base, i, val)
    assert v == len(snapshots)              # ids stay in lock-step with our snapshots
    snapshots.append(snap)

for v in range(len(snapshots)):             # every past version still queries correctly
    for _ in range(8):
        l = random.randint(0, n - 1); r = random.randint(l, n - 1)
        assert pst.range_sum(v, l, r) == sum(snapshots[v][l:r + 1])
print(f"{len(snapshots)} versions (random branching): every range-sum matches its snapshot")


151 versions (random branching): every range-sum matches its snapshot


## 5. When to use what

| You need… | Reach for | Why |
|---|---|---|
| which intervals cover a point / overlap a range | **interval tree** | O(log n + k); BSTs can't key on overlap |
| nearest point in 2-D / k-D | **k-d tree** | prunes the search; ~O(log n) average vs O(n) scan |
| all points inside an axis-aligned box | **k-d tree** range search | skips whole subtrees outside the box |
| query an array *as it was* at version k | **persistent segment tree** | O(log n) new nodes per version, old ones intact |
| stabbing many static intervals at once | sort endpoints + **sweep line** | one pass beats per-query work |
| point distances / convex hull / orientation | **computational geometry** | the geometric-primitives toolkit |

Two cautions. A k-d tree's pruning **degrades toward O(n)** as dimensionality climbs (the curse of dimensionality bites by ~10–20 D — use approximate methods like LSH or ball trees there). And persistence isn't free: see the cheat-sheet's memory note.

## 6. Complexity cheat-sheet

| Structure | Build | Query | Update | Extra memory |
|---|:---:|:---:|:---:|:---:|
| Interval tree | O(n log n) | O(log n + k) | — (static) | O(n) |
| k-d tree (2-D) | O(n log n) | O(log n + k) avg* | O(log n) amortized† | O(n) |
| Persistent segment tree | O(n) | O(log n) per version | O(log n) **new nodes** | O(log n) **per version** |

<sub>*k-d nearest/range is O(√n + k) worst case in 2-D and degrades toward O(n) in high dimensions. †k-d trees rebalance poorly under churn; rebuild periodically. **Memory note:** path copying makes each persistent update allocate only ~⌈log₂ n⌉ new nodes (shown above) — so *m* updates cost O(m log n) total memory, not O(mn). Cheap per version, but it still accumulates: holding many versions keeps all their paths alive, so long-lived persistent structures trade memory for time-travel.</sub>

These build on the segment tree from the **fenwick & segment trees** notebook, the balanced-tree intuition from **trees**, the one-pass interval idea in **sweep line**, and the distance/orientation primitives in **computational geometry**.